In [1]:
!pip install ultralytics roboflow -q

import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 8.3 MB/s eta 0:00:00
GPU: True Tesla T4


In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY_HERE")
visdrone_project = rf.workspace("ameeng").project("visdrone-detection-2")
visdrone_version = visdrone_project.version(1)
visdrone_dataset = visdrone_version.download("yolov8", location="/content/visdrone-raw")

print("VisDrone downloaded to:", visdrone_dataset.location)

upload and label your dataset, and get an API KEY here: https://app.roboflow.com/?model=undefined&ref=undefined
loading Roboflow workspace...


RoboflowError: {"error":{"message":"This API key does not exist (or has been revoked).","status":401,"type":"OAuthException","hint":"You may retrieve your API key via the Roboflow Dashboard. Go to Account > Roboflow Keys to retrieve yours."}}

In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key="eTyDkXF4Tex06U3nvtvp")
visdrone_project = rf.workspace("ameeng").project("visdrone-detection-2")
visdrone_version = visdrone_project.version(1)
visdrone_dataset = visdrone_version.download("yolov8", location="/content/visdrone-raw")

print("VisDrone downloaded to:", visdrone_dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/visdrone-raw in yolov8:: 100%|██████████| 17264/17264 [00:02<00:00, 6727.74it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
VisDrone downloaded to: /content/visdrone-raw


In [4]:
import yaml

with open("/content/visdrone-raw/data.yaml", "r") as f:
    vd_config = yaml.safe_load(f)

print("VisDrone classes:", vd_config['names'])
print("Number of classes:", vd_config['nc'])

VisDrone classes: ['awning-tricycle', 'bicycle', 'bus', 'car', 'motor', 'other', 'pedestrian', 'people', 'tricycle', 'truck', 'van']
Number of classes: 11


In [5]:
NEW_CLASSES = {0: "drone", 1: "car", 2: "person", 3: "truck", 4: "van"}

In [6]:
import os, shutil, yaml, glob

# === CONFIG ===
DRONE_DIR = "/content/drones-detect-1"
VISDRONE_DIR = "/content/visdrone-raw"
MERGED_DIR = "/content/merged-aerial"

# 5 unified classes
NEW_CLASSES = {0: "drone", 1: "car", 2: "person", 3: "truck", 4: "van"}

# Read VisDrone's original class list
with open(f"{VISDRONE_DIR}/data.yaml") as f:
    vd_cfg = yaml.safe_load(f)
vd_names = vd_cfg['names']

# Map VisDrone's class indices to our new indices (None = skip)
vd_remap = {}
for i, name in enumerate(vd_names):
    if name == "car":
        vd_remap[i] = 1
    elif name in ("pedestrian", "people"):
        vd_remap[i] = 2
    elif name == "truck":
        vd_remap[i] = 3
    elif name == "van":
        vd_remap[i] = 4
    else:
        vd_remap[i] = None

print("VisDrone remap:", {vd_names[k]: v for k, v in vd_remap.items()})

# Create merged directory structure
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_DIR}/{split}/labels", exist_ok=True)

def copy_drone_data(src_dir, merged_dir):
    copied = 0
    for split in ["train", "valid", "test"]:
        img_dir = f"{src_dir}/{split}/images"
        lbl_dir = f"{src_dir}/{split}/labels"
        if not os.path.exists(img_dir):
            continue
        for img_file in glob.glob(f"{img_dir}/*"):
            basename = os.path.basename(img_file)
            name_no_ext = os.path.splitext(basename)[0]
            new_name = f"drone_{basename}"
            shutil.copy(img_file, f"{merged_dir}/{split}/images/{new_name}")
            lbl_file = f"{lbl_dir}/{name_no_ext}.txt"
            if os.path.exists(lbl_file):
                shutil.copy(lbl_file, f"{merged_dir}/{split}/labels/drone_{name_no_ext}.txt")
                copied += 1
    return copied

def copy_visdrone_data(src_dir, merged_dir, remap):
    copied = 0
    skipped_images = 0
    for split in ["train", "valid", "test"]:
        img_dir = f"{src_dir}/{split}/images"
        lbl_dir = f"{src_dir}/{split}/labels"
        if not os.path.exists(img_dir):
            continue
        for img_file in glob.glob(f"{img_dir}/*"):
            basename = os.path.basename(img_file)
            name_no_ext = os.path.splitext(basename)[0]
            lbl_file = f"{lbl_dir}/{name_no_ext}.txt"
            if not os.path.exists(lbl_file):
                continue
            new_lines = []
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    old_cls = int(parts[0])
                    new_cls = remap.get(old_cls)
                    if new_cls is not None:
                        new_lines.append(f"{new_cls} {' '.join(parts[1:])}")
            if new_lines:
                new_name = f"vd_{basename}"
                shutil.copy(img_file, f"{merged_dir}/{split}/images/{new_name}")
                with open(f"{merged_dir}/{split}/labels/vd_{name_no_ext}.txt", "w") as f:
                    f.write("\n".join(new_lines) + "\n")
                copied += 1
            else:
                skipped_images += 1
    return copied, skipped_images

print("\nCopying drone dataset...")
drone_count = copy_drone_data(DRONE_DIR, MERGED_DIR)
print(f"  Drone images copied: {drone_count}")

print("Copying VisDrone dataset (filtering to car/person/truck/van)...")
vd_count, vd_skipped = copy_visdrone_data(VISDRONE_DIR, MERGED_DIR, vd_remap)
print(f"  VisDrone images kept: {vd_count} (skipped {vd_skipped} with no target classes)")

# Write merged data.yaml
merged_yaml = {
    "train": f"{MERGED_DIR}/train/images",
    "val": f"{MERGED_DIR}/valid/images",
    "test": f"{MERGED_DIR}/test/images",
    "nc": 5,
    "names": ["drone", "car", "person", "truck", "van"]
}
with open(f"{MERGED_DIR}/data.yaml", "w") as f:
    yaml.dump(merged_yaml, f)

# Count final images per split
for split in ["train", "valid", "test"]:
    n = len(glob.glob(f"{MERGED_DIR}/{split}/images/*"))
    print(f"  {split}: {n} images")

print(f"\nMerged dataset ready at {MERGED_DIR}")
print(f"Classes: {NEW_CLASSES}")

VisDrone remap: {'awning-tricycle': None, 'bicycle': None, 'bus': None, 'car': 1, 'motor': None, 'other': None, 'pedestrian': 2, 'people': 2, 'tricycle': None, 'truck': 3, 'van': 4}

Copying drone dataset...
  Drone images copied: 0
Copying VisDrone dataset (filtering to car/person/truck/van)...
  VisDrone images kept: 8626 (skipped 0 with no target classes)
  train: 6469 images
  valid: 547 images
  test: 1610 images

Merged dataset ready at /content/merged-aerial
Classes: {0: 'drone', 1: 'car', 2: 'person', 3: 'truck', 4: 'van'}


In [7]:
import os

# Check what's actually inside the drone dataset folder
print("Drone dataset contents:")
for item in os.listdir("/content/drones-detect-1"):
    subpath = f"/content/drones-detect-1/{item}"
    if os.path.isdir(subpath):
        print(f"  {item}/")
        for sub in os.listdir(subpath):
            subsubpath = f"{subpath}/{sub}"
            if os.path.isdir(subsubpath):
                count = len(os.listdir(subsubpath))
                print(f"    {sub}/ ({count} files)")
    else:
        print(f"  {item}")

Drone dataset contents:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drones-detect-1'

In [8]:
from roboflow import Roboflow

rf = Roboflow(api_key="eTyDkXF4Tex06U3nvtvp")
project = rf.workspace("keypointdetection-bwrv7").project("drones-detect-qhrmt")
version = project.version(1)
dataset = version.download("yolov8", location="/content/drones-detect-1")

print("Downloaded to:", dataset.location)

# Check structure
import os
for item in os.listdir("/content/drones-detect-1"):
    subpath = f"/content/drones-detect-1/{item}"
    if os.path.isdir(subpath):
        for sub in os.listdir(subpath):
            subsubpath = f"{subpath}/{sub}"
            if os.path.isdir(subsubpath):
                print(f"  {item}/{sub}/ ({len(os.listdir(subsubpath))} files)")
    else:
        print(f"  {item}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drones-detect-1 in yolov8:: 100%|██████████| 14415/14415 [00:09<00:00, 1501.89it/s]


Downloaded to: /content/drones-detect-1
  train/images/ (7041 files)
  train/labels/ (7041 files)
  valid/images/ (135 files)
  valid/labels/ (135 files)
  README.dataset.txt
  test/images/ (29 files)
  test/labels/ (29 files)
  data.yaml
  README.roboflow.txt


In [9]:
import shutil, glob

drone_count = 0
for split in ["train", "valid", "test"]:
    img_dir = f"/content/drones-detect-1/{split}/images"
    lbl_dir = f"/content/drones-detect-1/{split}/labels"
    for img_file in glob.glob(f"{img_dir}/*"):
        basename = os.path.basename(img_file)
        name_no_ext = os.path.splitext(basename)[0]
        shutil.copy(img_file, f"/content/merged-aerial/{split}/images/drone_{basename}")
        lbl_file = f"{lbl_dir}/{name_no_ext}.txt"
        if os.path.exists(lbl_file):
            shutil.copy(lbl_file, f"/content/merged-aerial/{split}/labels/drone_{name_no_ext}.txt")
            drone_count += 1

print(f"Drone images added: {drone_count}")

# Final counts
for split in ["train", "valid", "test"]:
    n = len(glob.glob(f"/content/merged-aerial/{split}/images/*"))
    print(f"  {split}: {n} images")

Drone images added: 7205
  train: 13510 images
  valid: 682 images
  test: 1639 images


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/merged-aerial/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    name="aerial_multiclass_v1"
)

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged-aerial/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aerial_multiclass_v1, nbs=64, nms=None, opse

In [12]:
import shutil, os, glob

os.makedirs("/content/portfolio_export_v2", exist_ok=True)

# Best weights
shutil.copy("/content/runs/detect/aerial_multiclass_v1/weights/best.pt", "/content/portfolio_export_v2/best.pt")

# Training plots
for f in ["confusion_matrix.png", "results.png", "PR_curve.png"]:
    src = f"/content/runs/detect/aerial_multiclass_v1/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/portfolio_export_v2/{f}")

# Val prediction images
for img in glob.glob("/content/runs/detect/aerial_multiclass_v1/val_batch*_pred.jpg")[:2]:
    shutil.copy(img, f"/content/portfolio_export_v2/{os.path.basename(img)}")

print("Files:", os.listdir("/content/portfolio_export_v2"))

Files: ['val_batch1_pred.jpg', 'val_batch2_pred.jpg', 'confusion_matrix.png', 'best.pt', 'results.png']


In [13]:
!zip -r /content/portfolio_export_v2.zip /content/portfolio_export_v2

from google.colab import files
files.download('/content/portfolio_export_v2.zip')

  adding: content/portfolio_export_v2/ (stored 0%)
  adding: content/portfolio_export_v2/val_batch1_pred.jpg (deflated 12%)
  adding: content/portfolio_export_v2/val_batch2_pred.jpg (deflated 4%)
  adding: content/portfolio_export_v2/confusion_matrix.png (deflated 22%)
  adding: content/portfolio_export_v2/best.pt (deflated 9%)
  adding: content/portfolio_export_v2/results.png (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>